In [17]:
#dissolve nonzero detections by event_id
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\nonzero_detections_dissolved_by_event"

arcpy.management.Dissolve(
    in_features=in_fc,
    out_feature_class=out_fc,
    dissolve_field=["event_id"],
    statistics_fields=[
        ["prebd_min_corrected", "MIN"],
        ["bd_min_corrected_plus8", "MAX"],
        ["area_ha", "SUM"],
        ["detection_id", "COUNT"]
    ],
    multi_part="MULTI_PART"
)

<Result 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\ClassiFIRE.gdb\\nonzero_detections_dissolved_by_event'>

In [18]:
#paths
import arcpy
from datetime import datetime

# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
events = f"{gdb}\\nonzero_detections_dissolved_by_event"
fod = f"{gdb}\\FOD_v7_SE_clip_1994_2022_smaller_than_0_81ha_removed"

arcpy.env.workspace = gdb
arcpy.env.scratchWorkspace = gdb



In [19]:
# -------------------------------------------------------------------
# DATE PARSERS
# -------------------------------------------------------------------

def parse_sefm_date(value):
    """Convert SEFM YYYYMMDD integer/string into datetime."""
    if value is None:
        return None
    return datetime.strptime(str(value), "%Y%m%d")

def parse_fod_date(value):
    """Convert FOD MM/DD/YYYY string into datetime."""
    if value is None:
        return None
    return datetime.strptime(value, "%m/%d/%Y")


In [20]:
#add classification fields
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
events = f"{gdb}\\nonzero_detections_dissolved_by_event"

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]

for b in buffer_sizes:
    field = f"buffer_{b}"
    if field not in [f.name for f in arcpy.ListFields(events)]:
        arcpy.management.AddField(events, field, "TEXT", field_length=20)

In [21]:
#buffer fod at each distance

buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

for label, dist in buffer_distances.items():
    out_fc = f"{gdb}\\FOD_buffer_{label}"
    arcpy.analysis.Buffer(
        in_features=fod,
        out_feature_class=out_fc,
        buffer_distance_or_field=dist,
        dissolve_option="NONE"
    )

In [24]:
#update parser
def parse_fod_date(value):
    if isinstance(value, datetime):
        return value
    return datetime.strptime(value, "%m/%d/%Y")

In [25]:
# BUILD FOD DATE LOOKUP
# -------------------------------------------------------------------

fod_dates = {}

with arcpy.da.SearchCursor(fod, ["FOD_ID", "DISCOVERY_DATE"]) as cur:
    for fid, d in cur:
        fod_dates[fid] = parse_fod_date(d)


In [26]:
#SPATIAL + TEMPORAL MATCHING FOR EACH BUFFER SIZE
# -------------------------------------------------------------------

for label in buffer_distances.keys():

    print(f"Processing buffer {label}...")

    buffer_fc = f"{gdb}\\FOD_buffer_{label}"

    # Spatial join output forced into working GDB
    sj = f"{gdb}\\sj_{label}"

    arcpy.analysis.SpatialJoin(
        target_features=events,
        join_features=buffer_fc,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # Build mapping: event_id → list of FOD_IDs intersecting it
    event_to_fod = {}

    with arcpy.da.SearchCursor(sj, ["event_id", "FOD_ID"]) as cur:
        for eid, fid in cur:
            if fid is None:
                continue
            event_to_fod.setdefault(eid, []).append(fid)

    # Update classification field
    field = f"buffer_{label}"

    with arcpy.da.UpdateCursor(
        events,
        ["event_id", "MIN_prebd_min_corrected",
         "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for eid, tmin_raw, tmax_raw, _ in cur:

            # Parse SEFM dates
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            classification = "prescribed"  # default

            # If event has spatial matches
            if eid in event_to_fod:

                # Check temporal match for ANY matched FOD report
                for fid in event_to_fod[eid]:
                    fod_date = fod_dates.get(fid)

                    if fod_date is not None and tmin <= fod_date <= tmax:
                        classification = "wildfire"
                        break

            # Write classification
            cur.updateRow([eid, tmin_raw, tmax_raw, classification])

print("Classification complete.")


Processing buffer 0_5km...
Processing buffer 1km...
Processing buffer 1_5km...
Processing buffer 2km...
Processing buffer 2_5km...
Processing buffer 3km...
Processing buffer 3_5km...
Processing buffer 4km...
Classification complete.


In [53]:
#at each buffer size, what % of fod are matched to events?
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
fod = f"{gdb}\\FOD_fires_clipped_to_SEFM_extent_1994_2020_undetectable_removed"

total_fod = int(arcpy.management.GetCount(fod)[0])
print("Total FOD reports:", total_fod)

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]
results = {}

for label in buffer_sizes:

    print(f"Processing {label}...")

    buffer_fc = f"{gdb}\\FOD_buffer_{label}"

    # Force spatial join output into your working GDB
    sj = f"{gdb}\\fod_match_{label}"

    # Overwrite if it already exists
    if arcpy.Exists(sj):
        arcpy.management.Delete(sj)

    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=f"{gdb}\\SEFM_events_dissolved",
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    matched_fod_ids = set()
    class_field = f"buffer_{label}"

    with arcpy.da.SearchCursor(sj, ["FOD_ID", class_field]) as cur:
        for fid, classification in cur:
            if classification == "wildfire":
                matched_fod_ids.add(fid)

    pct = (len(matched_fod_ids) / total_fod) * 100
    results[label] = pct

    print(f"{label}: {pct:.2f}% matched")

Total FOD reports: 360674
Processing 0_5km...
0_5km: 29.09% matched
Processing 1km...
1km: 51.83% matched
Processing 1_5km...
1_5km: 68.97% matched
Processing 2km...
2km: 80.40% matched
Processing 2_5km...
2_5km: 87.53% matched
Processing 3km...
3km: 92.02% matched


In [27]:
#it seems like that worked. Now go back and see if you can match those date =0 detections by year
import arcpy

# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

zero_date = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_0"
fod = f"{gdb}\\FOD_v7_SE_clip_1994_2022_smaller_than_0_81ha_removed"

arcpy.env.workspace = gdb
arcpy.env.scratchWorkspace = gdb

# -------------------------------------------------------------------
# 1. ADD YEAR-BASED CLASSIFICATION FIELDS
# -------------------------------------------------------------------

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]
year_fields = [f"buffer_{b}_year" for b in buffer_sizes]

existing = [f.name for f in arcpy.ListFields(zero_date)]

for fld in year_fields:
    if fld not in existing:
        arcpy.management.AddField(zero_date, fld, "TEXT", field_length=20)

# -------------------------------------------------------------------
# 2. BUILD FOD YEAR LOOKUP
# -------------------------------------------------------------------

fod_years = {}

with arcpy.da.SearchCursor(fod, ["FOD_ID", "FIRE_YEAR"]) as cur:
    for fid, yr in cur:
        fod_years[fid] = yr

# -------------------------------------------------------------------
# 3. YEAR-ONLY SPATIAL + TEMPORAL MATCHING
# -------------------------------------------------------------------

for label in buffer_sizes:

    print(f"Processing year-only classification for {label}...")

    buffer_fc = f"{gdb}\\FOD_buffer_{label}"
    sj = f"{gdb}\\zero_date_sj_{label}"

    # Overwrite if exists
    if arcpy.Exists(sj):
        arcpy.management.Delete(sj)

    # Spatial join: zero-date detections → FOD buffers
    arcpy.analysis.SpatialJoin(
        target_features=zero_date,
        join_features=buffer_fc,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # Build mapping: detection_id → list of FOD_IDs
    det_to_fod = {}

    with arcpy.da.SearchCursor(sj, ["detection_id", "FOD_ID"]) as cur:
        for det_id, fid in cur:
            if fid is None:
                continue
            det_to_fod.setdefault(det_id, []).append(fid)

    # Update classification field
    field = f"buffer_{label}_year"

    with arcpy.da.UpdateCursor(zero_date, ["detection_id", "year", field]) as cur:
        for det_id, det_year, _ in cur:

            classification = "prescribed"

            if det_id in det_to_fod:
                for fid in det_to_fod[det_id]:
                    fod_year = fod_years.get(fid)

                    if fod_year == det_year:
                        classification = "wildfire"
                        break

            cur.updateRow([det_id, det_year, classification])

print("Year-only classification complete.")

Processing year-only classification for 0_5km...
Processing year-only classification for 1km...
Processing year-only classification for 1_5km...
Processing year-only classification for 2km...
Processing year-only classification for 2_5km...
Processing year-only classification for 3km...
Processing year-only classification for 3_5km...
Processing year-only classification for 4km...
Year-only classification complete.


In [33]:
#merge full date and zero date into one layer
import arcpy

# -------------------------------------------------------------------
# PATHS AND SETTINGS
# -------------------------------------------------------------------
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

events    = f"{gdb}\\nonzero_detections_dissolved_by_event"
zero_date = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_0"
merged    = f"{gdb}\\SEFM_events_94_22_complete"

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]

# -------------------------------------------------------------------
# 1. ENSURE ZERO-DATE DETECTIONS HAVE EVENT_ID
#    (assign sequential IDs where event_id is NULL)
# -------------------------------------------------------------------

# Make sure event_id exists on zero_date
field_names = [f.name for f in arcpy.ListFields(zero_date)]
if "event_id" not in field_names:
    arcpy.management.AddField(zero_date, "event_id", "TEXT", field_length=50)

# Build year → counter map for zero-date rows with NULL event_id
year_counters = {}

with arcpy.da.SearchCursor(zero_date, ["event_id", "year"]) as cur:
    for eid, yr in cur:
        if eid is None and yr is not None:
            if yr not in year_counters:
                year_counters[yr] = 1

# Assign sequential event_id to zero-date detections
with arcpy.da.UpdateCursor(zero_date, ["event_id", "year"]) as cur:
    for eid, yr in cur:
        if eid is None and yr is not None:
            counter = year_counters[yr]
            new_id = f"Z{yr}_{counter:04d}"
            cur.updateRow([new_id, yr])
            year_counters[yr] += 1

print("Zero-date detections now have sequential event_id values.")

# -------------------------------------------------------------------
# 2. REBUILD SEFM_EVENTS_COMPLETE = EVENTS + ZERO-DATE DETECTIONS
# -------------------------------------------------------------------

if arcpy.Exists(merged):
    arcpy.management.Delete(merged)

# Merge event-level SEFM events and zero-date detections
arcpy.management.Merge([events, zero_date], merged)
print("SEFM_events_complete rebuilt via Merge(events, zero_date).")

# -------------------------------------------------------------------
# 3. ADD FINAL CLASSIFICATION FIELDS
# -------------------------------------------------------------------

for b in buffer_sizes:
    final = f"buffer_{b}_final"
    if final not in [f.name for f in arcpy.ListFields(merged)]:
        arcpy.management.AddField(merged, final, "TEXT", field_length=20)

print("Final classification fields added to SEFM_events_complete.")

# -------------------------------------------------------------------
# 4. BUILD LOOKUP: event_id → YEAR-ONLY CLASSIFICATIONS (FROM ZERO-DATE)
# -------------------------------------------------------------------

year_fields = [f"buffer_{b}_year" for b in buffer_sizes]
zero_lookup = {}

with arcpy.da.SearchCursor(zero_date, ["event_id"] + year_fields) as cur:
    for row in cur:
        eid = row[0]
        zero_lookup[eid] = row[1:]  # list of year-only classifications

print(f"Zero-date lookup built for {len(zero_lookup)} event_ids.")

# -------------------------------------------------------------------
# 5. POPULATE _FINAL FIELDS IN SEFM_EVENTS_COMPLETE
#    RULE:
#      - If full-date classification exists → use it
#      - Else if zero-date year-only exists → use that
#      - Else → leave as None (or 'prescribed' if you prefer)
# -------------------------------------------------------------------

for b in buffer_sizes:

    full  = f"buffer_{b}"
    final = f"buffer_{b}_final"

    idx = buffer_sizes.index(b)  # index into year-only list

    with arcpy.da.UpdateCursor(merged, ["event_id", full, final]) as cur:
        for eid, full_val, final_val in cur:

            if full_val in ("wildfire", "prescribed"):
                out = full_val
            else:
                year_vals = zero_lookup.get(eid)
                if year_vals:
                    out = year_vals[idx]
                else:
                    out = final_val  # leave as-is (often None or prescribed)

            cur.updateRow([eid, full_val, out])

print("Final harmonized classifications populated in SEFM_events_complete.")



Zero-date detections now have sequential event_id values.
SEFM_events_complete rebuilt via Merge(events, zero_date).
Final classification fields added to SEFM_events_complete.
Zero-date lookup built for 5666 event_ids.
Final harmonized classifications populated in SEFM_events_complete.


In [34]:
#Add event year from event id:
import arcpy

fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_complete"

# Add field if missing
fields = [f.name for f in arcpy.ListFields(fc)]
if "event_year" not in fields:
    arcpy.management.AddField(fc, "event_year", "LONG")

with arcpy.da.UpdateCursor(fc, ["event_id", "event_year"]) as cur:
    for eid, yr in cur:
        if eid.startswith(("S", "Z")):
            year = int(eid[1:5])
        else:
            year = int(eid[0:4])
        cur.updateRow([eid, year])

In [35]:
# Now how many match FOD?
fod = f"{gdb}\\FOD_v7_SE_clip_1994_2022_smaller_than_0_81ha_removed"
total_fod = int(arcpy.management.GetCount(fod)[0])

results = {}

for b in buffer_sizes:

    print(f"Processing {b}...")

    buffer_fc = f"{gdb}\\FOD_buffer_{b}"
    sj = f"{gdb}\\complete_sj_{b}"

    if arcpy.Exists(sj):
        arcpy.management.Delete(sj)

    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=merged,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    matched = set()
    final_field = f"buffer_{b}_final"

    with arcpy.da.SearchCursor(sj, ["FOD_ID", final_field]) as cur:
        for fid, val in cur:
            if val == "wildfire":
                matched.add(fid)

    pct = (len(matched) / total_fod) * 100
    results[b] = pct

    print(f"{b}: {pct:.2f}% matched")

Processing 0_5km...
0_5km: 31.70% matched
Processing 1km...
1km: 54.36% matched
Processing 1_5km...
1_5km: 70.76% matched
Processing 2km...
2km: 81.56% matched
Processing 2_5km...
2_5km: 88.26% matched
Processing 3km...
3km: 92.47% matched
Processing 3_5km...
3_5km: 95.10% matched
Processing 4km...
4km: 96.76% matched


In [36]:
#Put area_ha in one column instead of across two:
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
merged = f"{gdb}\\SEFM_events_94_22_complete"

with arcpy.da.UpdateCursor(merged, ["area_ha", "SUM_area_ha"]) as cur:
    for area, sum_area in cur:
        if area is None and sum_area is not None:
            # area_ha gets the SUM_area_ha value
            cur.updateRow([sum_area, sum_area])

In [38]:
#remove redundant field
arcpy.management.DeleteField(merged, ["SUM_area_ha"])

<Result 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\ClassiFIRE.gdb\\SEFM_events_94_22_complete'>

In [37]:
#make COUNT_detection_id 1 where it is NULL (zero date detections)
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
merged = f"{gdb}\\SEFM_events_94_22_complete"

with arcpy.da.UpdateCursor(merged, ["COUNT_detection_id"]) as cur:
    for (count_val,) in cur:
        if count_val is None:
            cur.updateRow([1])

In [63]:
#Look at FOD reports that did not match anything at 3 km

import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\Rx_ClassiFIRE.gdb"

# Inputs
fod_buffer = f"{gdb}\\FOD_buffer_3km"
events = f"{gdb}\\SEFM_events_complete"

# Output
sj = f"{gdb}\\FOD_to_SEFM_3km_sj"
unmatched = f"{gdb}\\FOD_unmatched_3km"

# Clean up if needed
for fc in [sj, unmatched]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# 1. Spatial join: FOD buffer → SEFM events
arcpy.analysis.SpatialJoin(
    target_features=fod_buffer,
    join_features=events,
    out_feature_class=sj,
    join_operation="JOIN_ONE_TO_MANY",
    match_option="INTERSECT"
)

# 2. Build a set of FOD IDs that matched wildfire events
matched_ids = set()

with arcpy.da.SearchCursor(sj, ["FOD_ID", "buffer_3km_final"]) as cur:
    for fod_id, final_class in cur:
        if final_class == "wildfire":
            matched_ids.add(fod_id)

# 3. Select FOD reports that did NOT match
with arcpy.da.SearchCursor(fod_buffer, ["FOD_ID"]) as cur:
    all_ids = {row[0] for row in cur}

unmatched_ids = all_ids - matched_ids

# 4. Export unmatched FOD reports
arcpy.management.MakeFeatureLayer(fod_buffer, "fod_lyr")
arcpy.management.SelectLayerByAttribute(
    "fod_lyr",
    "NEW_SELECTION",
    f"FOD_ID IN ({','.join(str(i) for i in unmatched_ids)})"
)

arcpy.management.CopyFeatures("fod_lyr", unmatched)

print(f"Unmatched FOD reports: {len(unmatched_ids)}")
print(f"Output saved to: {unmatched}")

Unmatched FOD reports: 28698
Output saved to: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\Rx_ClassiFIRE.gdb\FOD_unmatched_3km


In [1]:
#subset FOD unmatched at 3km
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\Rx_ClassiFIRE.gdb"
unmatched_buffers = f"{gdb}\\FOD_unmatched_3km"
fod_points = f"{gdb}\\FOD_fires_clipped_to_SEFM_extent_1994_2020_undetectable_removed"
fod_id_field = "FOD_ID"   # <-- update if your ID field is named differently

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

# 1. Extract unique FOD IDs from the unmatched buffer layer
ids = set()
with arcpy.da.SearchCursor(unmatched_buffers, [fod_id_field]) as cursor:
    for row in cursor:
        if row[0] is not None:
            ids.add(row[0])

# 2. Build SQL IN clause
id_list = ",".join(str(i) for i in ids)
where = f"{fod_id_field} IN ({id_list})"

# 3. Select the actual FOD points
selection = arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=fod_points,
    selection_type="NEW_SELECTION",
    where_clause=where
)

# 4. Export the selected FOD reports
out_fc = f"{gdb}\\FOD_unmatched_reports3km"
arcpy.management.CopyFeatures(selection, out_fc)

print(f"Exported {len(ids)} unmatched FOD reports to {out_fc}")

Exported 28698 unmatched FOD reports to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\Rx_ClassiFIRE.gdb\FOD_unmatched_reports3km


In [40]:
#try to streamline the above two: (delete other two if it runs ok)
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

# Inputs
fod_buffer = f"{gdb}\\FOD_buffer_2_5km"   # 2.5 km FOD buffers
events = f"{gdb}\\SEFM_events_94_22_complete"  # final classified events
fod_points = f"{gdb}\\FOD_v7_SE_clip_1994_2022_smaller_than_0_81ha_removed"

# Output
out_points = f"{gdb}\\FOD_unmatched_reports_2_5km"

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

# Temporary spatial join output
sj = f"{gdb}\\_tmp_FOD_SEFM_2_5km_sj"
if arcpy.Exists(sj):
    arcpy.management.Delete(sj)

# 1. Spatial join: FOD buffers → SEFM events
arcpy.analysis.SpatialJoin(
    target_features=fod_buffer,
    join_features=events,
    out_feature_class=sj,
    join_operation="JOIN_ONE_TO_MANY",
    match_option="INTERSECT"
)

# 2. Build a set of FOD IDs that matched wildfire events
matched_ids = set()

with arcpy.da.SearchCursor(sj, ["FOD_ID", "buffer_2_5km_final"]) as cur:
    for fod_id, final_class in cur:
        if final_class == "wildfire":
            matched_ids.add(fod_id)

# 3. Get all FOD IDs from the original FOD points
all_ids = set()
with arcpy.da.SearchCursor(fod_points, ["FOD_ID"]) as cur:
    for row in cur:
        all_ids.add(row[0])

# 4. Compute unmatched IDs
unmatched_ids = all_ids - matched_ids

# 5. Select and export unmatched FOD points
if unmatched_ids:
    id_list = ",".join(str(i) for i in unmatched_ids)
    where = f"FOD_ID IN ({id_list})"

    arcpy.management.MakeFeatureLayer(fod_points, "fod_pts_lyr")
    arcpy.management.SelectLayerByAttribute("fod_pts_lyr", "NEW_SELECTION", where)
    arcpy.management.CopyFeatures("fod_pts_lyr", out_points)

    print(f"Exported {len(unmatched_ids)} unmatched FOD reports to {out_points}")
else:
    print("All FOD reports matched wildfire events at 2.5 km.")

# 6. Clean up temporary join
arcpy.management.Delete(sj)

Exported 37445 unmatched FOD reports to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\FOD_unmatched_reports_2_5km


<Result 'true'>